In [3]:
import pandas as pd
import numpy as np
import re
import string
from pathlib import Path
import unicodedata
from time import perf_counter

In [4]:
BASE_DIR = Path("..")

TRAIN_DIR = BASE_DIR / "data" / "train"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

print(S1_PATH)
print(S2_PATH)
print(S3_PATH)
print(GT_PATH)

..\data\train\train_source1.tsv
..\data\train\train_source2.tsv
..\data\train\train_source3.tsv
..\data\train\train_ground_truth.tsv


In [5]:
s1_sample = pd.read_csv(
    S1_PATH,
    sep="\t",
    nrows=10
)

s2_sample = pd.read_csv(
    S2_PATH,
    sep="\t",
    nrows=10
)

s3_sample = pd.read_csv(
    S3_PATH,
    sep="\t",
    nrows=10
)

In [6]:
s1_sample

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US


In [7]:
s2_sample

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
5,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US
6,S2-584977605,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",India
7,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India
8,S2-721031885,Chavira Platinum Chimera LLC,"282 SAXONY DRIVE, FTT MITCHELL, KY",US
9,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",India


In [8]:
s3_sample

,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India
5,S3-578159284,LLC Hernandez Colonial Redwood,"2260- Housecreek Trail, Unit 407, Raleigh, Nor...",US
6,S3-121412624,Classic Equity Partners Group,"6885 Catalpa Bluff Ln, PO Box 5799, Dickinson,...",US
7,S3-249416830,Ectolumdrex dba X+ Madison Inc,"S03575 Cty Tk M, Town Of Buffalo, WI",US
8,S3-107644605,Animal Hanisch Hospirlg,"##8 Willow Oak Lane, Fl. 0, Saint Louis, Missouri",US
9,S3-160217003,Gomez Optimal,"343 Hempstead 161, Hope, Arkansas",US


In [9]:
print("S1 columns:")
print(s1_sample.columns.tolist())

print("\nS2 columns:")
print(s2_sample.columns.tolist())

print("\nS3 columns:")
print(s3_sample.columns.tolist())

S1 columns:
['entity_id', 'business_name', 'business_address', 'country']

S2 columns:
['entity_id', 'business_name', 'business_address', 'country']

S3 columns:
['entity_id', 'business_name', 'business_address', 'country']


In [10]:
s1_sample.iloc[0]

entity_id                                     S1-925783039
business_name                          Orelee's Barbershop
business_address    1795 Westchester Drive, High Point, NC
country                                                 US
Name: 0, dtype: object

In [11]:
SAMPLE_SIZE = 1000

s1_sample = pd.read_csv(
    S1_PATH,
    sep="\t",
    nrows=SAMPLE_SIZE
)

s2_sample = pd.read_csv(
    S2_PATH,
    sep="\t",
    nrows=SAMPLE_SIZE
)

s3_sample = pd.read_csv(
    S3_PATH,
    sep="\t",
    nrows=SAMPLE_SIZE
)

print("S1:", s1_sample.shape)
print("S2:", s2_sample.shape)
print("S3:", s3_sample.shape)

S1: (1000, 4)
S2: (1000, 4)
S3: (1000, 4)


In [12]:
for name, df in {
    "S1": s1_sample,
    "S2": s2_sample,
    "S3": s3_sample
}.items():

    print(f"\n{name}")
    print(df.isna().sum())


S1
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

S2
entity_id            0
business_name        0
business_address    35
country              0
dtype: int64

S3
entity_id            0
business_name        0
business_address    37
country              0
dtype: int64


In [13]:
def normalize_text(value):
    """
    Conservative normalization.

    - Handles missing values
    - Unicode normalization
    - Lowercase
    - Normalizes whitespace
    - Removes punctuation while preserving Unicode letters/numbers
    """
    if pd.isna(value):
        return ""

    value = str(value)

    # Normalize Unicode without deleting non-Latin scripts
    value = unicodedata.normalize("NFKC", value)

    # Lowercase
    value = value.lower()

    # Replace punctuation with spaces
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value).strip()

    return value

In [14]:
s1_sample["name_norm"] = s1_sample["business_name"].map(normalize_text)
s2_sample["name_norm"] = s2_sample["business_name"].map(normalize_text)
s3_sample["name_norm"] = s3_sample["business_name"].map(normalize_text)

In [15]:
s1_sample[["business_name", "name_norm"]].head(10)

,business_name,name_norm
0,Orelee's Barbershop,orelee s barbershop
1,Prime Money,prime money
2,B+ Retail Inc,b retail inc
3,Christ Chapel,christ chapel
4,Prabhav Business Center,prabhav business center
5,Custom Wealth Services LLC,custom wealth services llc
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited
7,Nexus Anchor Rain,nexus anchor rain
8,Moore Bitwise Inc,moore bitwise inc
9,Dermatology Green Medicine,dermatology green medicine


In [16]:
from collections import defaultdict

def build_name_index(df):
    index = defaultdict(list)

    for _, row in df.iterrows():
        name = row["name_norm"]

        if name:
            index[name].append(row["entity_id"])

    return index

In [17]:
s2_name_index = build_name_index(s2_sample)
s3_name_index = build_name_index(s3_sample)

print("S2 unique normalized names:", len(s2_name_index))
print("S3 unique normalized names:", len(s3_name_index))

S2 unique normalized names: 1000
S3 unique normalized names: 1000


In [18]:
def exact_name_candidates(s1_df, s2_index, s3_index):
    candidates = []

    for _, row in s1_df.iterrows():

        s1_id = row["entity_id"]
        name = row["name_norm"]

        if not name:
            continue

        # Search S2
        for candidate_id in s2_index.get(name, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": "exact_name"
            })

        # Search S3
        for candidate_id in s3_index.get(name, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": "exact_name"
            })

    return pd.DataFrame(candidates)

In [19]:
name_candidates = exact_name_candidates(
    s1_sample,
    s2_name_index,
    s3_name_index
)

name_candidates

,s1_entity_id,candidate_entity_id,source,block_type
0,S1-746273151,S3-538882911,S3,exact_name
1,S1-8092452,S3-551271002,S3,exact_name
2,S1-401074421,S3-303073926,S3,exact_name


In [20]:
pair = name_candidates.iloc[0]

s1_id = pair["s1_entity_id"]
candidate_id = pair["candidate_entity_id"]

print("S1 ID:", s1_id)
print("Candidate ID:", candidate_id)

S1 ID: S1-746273151
Candidate ID: S3-538882911


In [21]:
s1_match = s1_sample[
    s1_sample["entity_id"] == s1_id
]

s3_match = s3_sample[
    s3_sample["entity_id"] == candidate_id
]

display(
    s1_match[
        ["entity_id", "business_name", "business_address", "country", "name_norm"]
    ]
)

display(
    s3_match[
        ["entity_id", "business_name", "business_address", "country", "name_norm"]
    ]
)

,entity_id,business_name,business_address,country,name_norm
99,S1-746273151,Pediatric Dental Physicians Inc,"53 Laroche Lane, Hebron, ME",US,pediatric dental physicians inc


,entity_id,business_name,business_address,country,name_norm
646,S3-538882911,Pediatric Dental Physicians Inc.,NaN,US,pediatric dental physicians inc


In [22]:
print("Total candidate pairs:", len(name_candidates))

Total candidate pairs: 3


In [23]:
candidates_per_s1 = (
    name_candidates
    .groupby("s1_entity_id")
    .size()
)

print("S1 records with candidates:", len(candidates_per_s1))

if len(candidates_per_s1) > 0:
    print("Average:", candidates_per_s1.mean())
    print("Median:", candidates_per_s1.median())
    print("P95:", candidates_per_s1.quantile(0.95))
    print("P99:", candidates_per_s1.quantile(0.99))
    print("Maximum:", candidates_per_s1.max())

S1 records with candidates: 3
Average: 1.0
Median: 1.0
P95: 1.0
P99: 1.0
Maximum: 1


In [24]:
for _, pair in name_candidates.iterrows():
    print("=" * 70)
    print("S1:", pair["s1_entity_id"])
    print("Candidate:", pair["candidate_entity_id"])
    print("Source:", pair["source"])
    print("Block:", pair["block_type"])

    s1_row = s1_sample[
        s1_sample["entity_id"] == pair["s1_entity_id"]
    ]

    if pair["source"] == "S2":
        candidate_row = s2_sample[
            s2_sample["entity_id"] == pair["candidate_entity_id"]
        ]
    else:
        candidate_row = s3_sample[
            s3_sample["entity_id"] == pair["candidate_entity_id"]
        ]

    print("\nS1 record:")
    display(
        s1_row[
            ["entity_id", "business_name", "business_address", "country"]
        ]
    )

    print("\nCandidate record:")
    display(
        candidate_row[
            ["entity_id", "business_name", "business_address", "country"]
        ]
    )

S1: S1-746273151
Candidate: S3-538882911
Source: S3
Block: exact_name

S1 record:


,entity_id,business_name,business_address,country
99,S1-746273151,Pediatric Dental Physicians Inc,"53 Laroche Lane, Hebron, ME",US



Candidate record:


,entity_id,business_name,business_address,country
646,S3-538882911,Pediatric Dental Physicians Inc.,NaN,US


S1: S1-8092452
Candidate: S3-551271002
Source: S3
Block: exact_name

S1 record:


...

,entity_id,business_name,business_address,country
265,S3-551271002,Urban Nails,"Wilson Pike, Brentwood, Tennessee",US


S1: S1-401074421
Candidate: S3-303073926
Source: S3
Block: exact_name

S1 record:


,entity_id,business_name,business_address,country
968,S1-401074421,Chau Minerals LLC,"1620 Belmont Street, Unit B, Washington, DC",US



Candidate record:


,entity_id,business_name,business_address,country
895,S3-303073926,Chau Minerals LLC,"1620 Belmont St, Washingtont Ownship, District...",US


In [25]:
def tokenize(value):
    if not value:
        return []

    return value.split()

In [26]:
s1_sample["name_tokens"] = s1_sample["name_norm"].map(tokenize)
s2_sample["name_tokens"] = s2_sample["name_norm"].map(tokenize)
s3_sample["name_tokens"] = s3_sample["name_norm"].map(tokenize)

s1_sample[["business_name", "name_norm", "name_tokens"]].head(10)

,business_name,name_norm,name_tokens
0,Orelee's Barbershop,orelee s barbershop,"[orelee, s, barbershop]"
1,Prime Money,prime money,"[prime, money]"
2,B+ Retail Inc,b retail inc,"[b, retail, inc]"
3,Christ Chapel,christ chapel,"[christ, chapel]"
4,Prabhav Business Center,prabhav business center,"[prabhav, business, center]"
5,Custom Wealth Services LLC,custom wealth services llc,"[custom, wealth, services, llc]"
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"[consulting, nyasa, nursing, private, limited]"
7,Nexus Anchor Rain,nexus anchor rain,"[nexus, anchor, rain]"
8,Moore Bitwise Inc,moore bitwise inc,"[moore, bitwise, inc]"
9,Dermatology Green Medicine,dermatology green medicine,"[dermatology, green, medicine]"


In [27]:
def build_token_index(df):
    index = defaultdict(set)

    for _, row in df.iterrows():
        entity_id = row["entity_id"]

        for token in row["name_tokens"]:
            if token:
                index[token].add(entity_id)

    return index

s2_name_token_index = build_token_index(s2_sample)
s3_name_token_index = build_token_index(s3_sample)

print("Unique S2 name tokens:", len(s2_name_token_index))
print("Unique S3 name tokens:", len(s3_name_token_index))

Unique S2 name tokens: 1810
Unique S3 name tokens: 1760


In [28]:
for token in list(s2_name_token_index.keys())[:20]:
    print(token, "→", list(s2_name_token_index[token])[:5])

र → ['S2-510494370', 'S2-760590368', 'S2-735885703', 'S2-219128254', 'S2-91404065']
म → ['S2-510494370', 'S2-760590368', 'S2-735885703', 'S2-219128254', 'S2-91404065']
क → ['S2-714861139', 'S2-615967906', 'S2-751652919', 'S2-64510343', 'S2-343318003']
ट → ['S2-510494370', 'S2-760590368', 'S2-735885703', 'S2-219128254', 'S2-91404065']
ग → ['S2-169597264', 'S2-343318003', 'S2-631299191', 'S2-166376419', 'S2-228141367']
प → ['S2-510494370', 'S2-760590368', 'S2-735885703', 'S2-219128254', 'S2-91404065']
इव → ['S2-510494370', 'S2-735885703', 'S2-219128254', 'S2-91404065', 'S2-331922251']
ल → ['S2-510494370', 'S2-735885703', 'S2-219128254', 'S2-91404065', 'S2-331922251']
ड → ['S2-510494370', 'S2-735885703', 'S2-219128254', 'S2-91404065', 'S2-331922251']
holloway → ['S2-764573417']
peak → ['S2-836840386', 'S2-162336156', 'S2-888440125', 'S2-764573417', 'S2-623626120']
inc → ['S2-799199580', 'S2-321565255', 'S2-376460483', 'S2-735517191', 'S2-967974285']
seafood → ['S2-764573417']
आद → ['S2-63

In [29]:
def name_token_candidates(s1_df, s2_index, s3_index):
    candidates = []

    for _, row in s1_df.iterrows():

        s1_id = row["entity_id"]
        tokens = row["name_tokens"]

        matched_s2 = set()
        matched_s3 = set()

        for token in tokens:
            matched_s2.update(s2_index.get(token, set()))
            matched_s3.update(s3_index.get(token, set()))

        for candidate_id in matched_s2:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": "name_token"
            })

        for candidate_id in matched_s3:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": "name_token"
            })

    return pd.DataFrame(candidates)

In [30]:
token_candidates_df = name_token_candidates(
    s1_sample,
    s2_name_token_index,
    s3_name_token_index
)

print("Total token-block candidates:", len(token_candidates_df))

token_candidates_df.head(20)

Total token-block candidates: 155171


,s1_entity_id,candidate_entity_id,source,block_type
0,S1-925783039,S2-769808418,S2,name_token
1,S1-925783039,S2-34975790,S2,name_token
2,S1-925783039,S2-921776853,S2,name_token
3,S1-925783039,S2-41847464,S2,name_token
4,S1-925783039,S2-930194769,S2,name_token
5,S1-925783039,S2-414418413,S2,name_token
6,S1-925783039,S2-373730786,S2,name_token
7,S1-925783039,S2-54521194,S2,name_token
8,S1-925783039,S2-222674377,S2,name_token
9,S1-925783039,S2-531458823,S2,name_token


In [31]:
if len(token_candidates_df) > 0:

    token_counts = (
        token_candidates_df
        .groupby("s1_entity_id")
        .size()
    )

    print("Total candidate pairs:", len(token_candidates_df))
    print("S1 records with candidates:", len(token_counts))
    print("Average:", token_counts.mean())
    print("Median:", token_counts.median())
    print("P95:", token_counts.quantile(0.95))
    print("P99:", token_counts.quantile(0.99))
    print("Maximum:", token_counts.max())

else:
    print("No token candidates found.")

Total candidate pairs: 155171
S1 records with candidates: 982
Average: 158.0152749490835
Median: 176.0
P95: 341.0
P99: 359.18999999999994
Maximum: 387


In [32]:
token_frequency_s2 = {
    token: len(entity_ids)
    for token, entity_ids in s2_name_token_index.items()
}

token_frequency_s3 = {
    token: len(entity_ids)
    for token, entity_ids in s3_name_token_index.items()
}

In [33]:
print("\nMost common S3 tokens:")

for token, count in sorted(
    token_frequency_s3.items(),
    key=lambda x: x[1],
    reverse=True
)[:30]:
    print(repr(token), "→", count)


Most common S3 tokens:
'limited' → 132
'private' → 126
'llc' → 93
'inc' → 85
'ltd' → 72
'pvt' → 44
'com' → 43
'center' → 34
's' → 33
'partners' → 33
'services' → 30
'india' → 29
'ल' → 26
'ट' → 25
'l' → 25
'group' → 24
'c' → 24
'ड' → 23
'र' → 23
'म' → 23
'holdings' → 22
'co' → 20
'and' → 20
'care' → 19
'corp' → 19
'a' → 19
'lp' → 19
'प' → 19
'इव' → 17
'corporation' → 14


In [34]:
NAME_STOPWORDS = {
    "inc", "llc", "ltd", "pvt", "co", "corp", "corporation",
    "limited", "group", "company", "the", "and", "of",
}


def build_token_index(df, token_col, stopwords=None, max_doc_freq=None):
    """
    Build an inverted index: token -> list of entity_ids containing that token.

    stopwords: tokens to skip entirely (too generic to be useful).
    max_doc_freq: if set, any token appearing in MORE than this many records
                  is dropped from the index after counting (protects against
                  tokens we didn't think to add to the stopword list).
    """
    stopwords = stopwords or set()
    index = defaultdict(list)

    for _, row in df.iterrows():
        entity_id = row["entity_id"]
        tokens = row[token_col]

        for token in tokens:
            if len(token) < 2 or token in stopwords:
                continue
            index[token].append(entity_id)

    if max_doc_freq is not None:
        index = {
            token: ids
            for token, ids in index.items()
            if len(ids) <= max_doc_freq
        }

    return index


In [35]:
s2_name_token_index = build_token_index(s2_sample, "name_tokens", stopwords=NAME_STOPWORDS)
s3_name_token_index = build_token_index(s3_sample, "name_tokens", stopwords=NAME_STOPWORDS)

print("S2 name-token index size:", len(s2_name_token_index))
print("S3 name-token index size:", len(s3_name_token_index))

token_doc_freq = sorted(
    ((token, len(ids)) for token, ids in s2_name_token_index.items()),
    key=lambda x: -x[1],
)
print("\nMost frequent S2 name tokens:")
for token, freq in token_doc_freq[:15]:
    print(f"  {token!r}: {freq}")


S2 name-token index size: 1639
S3 name-token index size: 1603

Most frequent S2 name tokens:
  'private': 121
  'com': 47
  'center': 38
  'partners': 35
  'इव': 32
  'services': 25
  'care': 25
  'india': 25
  'holdings': 18
  'llp': 17
  'associates': 14
  'global': 13
  'industries': 13
  'brothers': 10
  'service': 10


In [36]:
def token_block_candidates(s1_df, token_col, s2_index, s3_index, block_type):

    candidates = []

    for _, row in s1_df.iterrows():
        s1_id = row["entity_id"]
        tokens = row[token_col]

        if not tokens:
            continue

        matched_s2 = set()
        matched_s3 = set()

        for token in tokens:
            matched_s2.update(s2_index.get(token, []))
            matched_s3.update(s3_index.get(token, []))

        for candidate_id in matched_s2:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": block_type,
            })

        for candidate_id in matched_s3:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": block_type,
            })

    return pd.DataFrame(candidates)


In [ ]:
name_token_candidates = token_block_candidates(
    s1_sample,
    "name_tokens",
    s2_name_token_index,
    s3_name_token_index,
    block_type="name_token",
)

print("Name-token candidate pairs:", len(name_token_candidates))
print("Exact-name candidate pairs (Strategy 1):", len(name_candidates))
name_token_candidates.head(10)


In [ ]:
def candidate_stats(candidates_df, label=""):

    if label:
        print(f"--- {label} ---")

    if len(candidates_df) == 0:
        print("No candidates generated.")
        return None

    per_s1 = candidates_df.groupby("s1_entity_id").size()

    print("S1 records with at least one candidate:", len(per_s1))
    print("Average candidates per S1:", per_s1.mean())
    print("Median candidates per S1:", per_s1.median())
    print("P95:", per_s1.quantile(0.95))
    print("P99:", per_s1.quantile(0.99))
    print("Max:", per_s1.max())

    return per_s1


_ = candidate_stats(name_token_candidates, label="Name-token block (sample)")


In [ ]:
def normalize_address(value):

    return normalize_text(value)


s1_sample["addr_norm"] = s1_sample["business_address"].map(normalize_address)
s2_sample["addr_norm"] = s2_sample["business_address"].map(normalize_address)
s3_sample["addr_norm"] = s3_sample["business_address"].map(normalize_address)

s1_sample[["business_address", "addr_norm"]].head(10)


In [ ]:
missing_addr_rows = s3_sample[s3_sample["business_address"].isna()]
missing_addr_rows[["entity_id", "business_name", "business_address", "addr_norm"]]

In [ ]:
s1_sample["addr_tokens"] = s1_sample["addr_norm"].map(tokenize)
s2_sample["addr_tokens"] = s2_sample["addr_norm"].map(tokenize)
s3_sample["addr_tokens"] = s3_sample["addr_norm"].map(tokenize)

s1_sample[["business_address", "addr_norm", "addr_tokens"]].head(10)


In [ ]:
ADDRESS_STOPWORDS = {
    "street", "st", "road", "rd", "avenue", "ave", "lane", "ln",
    "drive", "dr", "near", "opp", "no", "nagar",
    "north", "south", "east", "west",
}

s2_addr_token_index = build_token_index(s2_sample, "addr_tokens", stopwords=ADDRESS_STOPWORDS)
s3_addr_token_index = build_token_index(s3_sample, "addr_tokens", stopwords=ADDRESS_STOPWORDS)

print("S2 address-token index size:", len(s2_addr_token_index))
print("S3 address-token index size:", len(s3_addr_token_index))

In [ ]:
address_token_candidates = token_block_candidates(
    s1_sample,
    "addr_tokens",
    s2_addr_token_index,
    s3_addr_token_index,
    block_type="address_token",
)

print("Address-token candidate pairs:", len(address_token_candidates))
_ = candidate_stats(address_token_candidates, label="Address-token block (sample)")


In [ ]:
import re as _re  


def extract_leading_number(value):

    if not value:
        return None

    match = re.search(r"\d+", value)
    return match.group(0) if match else None


s1_sample["addr_number"] = s1_sample["addr_norm"].map(extract_leading_number)
s2_sample["addr_number"] = s2_sample["addr_norm"].map(extract_leading_number)
s3_sample["addr_number"] = s3_sample["addr_norm"].map(extract_leading_number)

s1_sample[["business_address", "addr_norm", "addr_number"]].head(10)

In [ ]:
def build_number_index(df, number_col, max_doc_freq=None):

    index = defaultdict(list)

    for _, row in df.iterrows():
        number = row[number_col]
        if number:
            index[number].append(row["entity_id"])

    if max_doc_freq is not None:
        index = {
            number: ids
            for number, ids in index.items()
            if len(ids) <= max_doc_freq
        }

    return index

MAX_NUMBER_DOC_FREQ = 5

s2_number_index = build_number_index(s2_sample, "addr_number", max_doc_freq=MAX_NUMBER_DOC_FREQ)
s3_number_index = build_number_index(s3_sample, "addr_number", max_doc_freq=MAX_NUMBER_DOC_FREQ)

print("S2 address-number index size:", len(s2_number_index))
print("S3 address-number index size:", len(s3_number_index))


In [ ]:
def address_number_candidates(s1_df, s2_index, s3_index):
   
    candidates = []

    for _, row in s1_df.iterrows():
        s1_id = row["entity_id"]
        number = row["addr_number"]

        if not number:
            continue

        for candidate_id in s2_index.get(number, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": "address_number",
            })

        for candidate_id in s3_index.get(number, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": "address_number",
            })

    return pd.DataFrame(candidates)


number_candidates = address_number_candidates(s1_sample, s2_number_index, s3_number_index)

print("Address-number candidate pairs:", len(number_candidates))
_ = candidate_stats(number_candidates, label="Address-number block (sample)")


In [ ]:
def combine_candidate_blocks(block_dfs):
   
    non_empty = [df for df in block_dfs if len(df) > 0]

    if not non_empty:
        return pd.DataFrame(
            columns=["s1_entity_id", "candidate_entity_id", "source", "retrieved_by"]
        )

    combined = pd.concat(non_empty, ignore_index=True)

    combined = (
        combined
        .groupby(["s1_entity_id", "candidate_entity_id", "source"])["block_type"]
        .agg(lambda block_types: sorted(set(block_types)))
        .reset_index()
        .rename(columns={"block_type": "retrieved_by"})
    )

    return combined


combined_candidates = combine_candidate_blocks([
    name_candidates,
    name_token_candidates,
    address_token_candidates,
    number_candidates,
])

total_raw = (
    len(name_candidates)
    + len(name_token_candidates)
    + len(address_token_candidates)
    + len(number_candidates)
)

print("Total candidate rows before dedup (sum across blocks):", total_raw)
print("Total unique candidate pairs after union + dedup:", len(combined_candidates))
combined_candidates.head(10)


In [ ]:
from collections import Counter

block_membership_counts = Counter()
for retrieved_by in combined_candidates["retrieved_by"]:
    for block_type in retrieved_by:
        block_membership_counts[block_type] += 1

print("How many combined candidate pairs each block contributed to:")
for block_type, count in block_membership_counts.most_common():
    print(f"  {block_type}: {count}")

multi_block_count = (combined_candidates["retrieved_by"].map(len) > 1).sum()
print(f"\nPairs retrieved by more than one block: {multi_block_count} / {len(combined_candidates)}")

combined_candidates[combined_candidates["retrieved_by"].map(len) > 1].head(10)

In [ ]:
_ = candidate_stats(combined_candidates, label="Combined blocks (sample, all 4 strategies)")

s1_with_candidates = set(combined_candidates["s1_entity_id"])
s1_without_candidates = set(s1_sample["entity_id"]) - s1_with_candidates

print(f"\nS1 records with at least one candidate: {len(s1_with_candidates)} / {len(s1_sample)}")
print(f"S1 records with NO candidates from any block: {len(s1_without_candidates)}")


In [ ]:
import json as _json

VALIDATION_DIR = BASE_DIR / "code" / "business_entity_resolution" / "src" / "validation_artifacts"

VAL_S1_IDS_PATH = VALIDATION_DIR / "val_s1_ids.txt"
VAL_GT_DICT_PATH = VALIDATION_DIR / "val_gt_dict.json"

print(VAL_S1_IDS_PATH)
print(VAL_GT_DICT_PATH)


In [ ]:
def load_val_s1_ids(path):
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def load_val_gt_dict(path):

    raw = path.read_text(encoding="utf-8", errors="replace")
    entries = re.findall(r'"(S1-[^"]+)":\s*\[(.*?)\]', raw, flags=re.DOTALL)

    val_gt = {}
    corrupted = []

    for s1_id, body in entries:
        raw_ids = re.findall(r'"([^"]*)"', body)
        cleaned = {i for i in raw_ids if i not in ("nan", "")}

        bad = [i for i in cleaned if not i.startswith(("S2-", "S3-"))]
        if bad:
            corrupted.append((s1_id, bad))

        val_gt[s1_id] = {i for i in cleaned if i.startswith(("S2-", "S3-"))}

    return val_gt, corrupted


val_s1_ids = load_val_s1_ids(VAL_S1_IDS_PATH)
val_gt_dict, corrupted_entries = load_val_gt_dict(VAL_GT_DICT_PATH)

print("Validation S1 IDs:", len(val_s1_ids))
print("Ground-truth entries loaded:", len(val_gt_dict))
print("Corrupted entries found:", len(corrupted_entries))

missing_from_dict = [s1_id for s1_id in val_s1_ids if s1_id not in val_gt_dict]
print(f"Validation IDs with NO ground-truth entry at all: {len(missing_from_dict)}")
if missing_from_dict:
    print("  (first 5):", missing_from_dict[:5])


In [ ]:
def compute_candidate_recall(candidates_df, val_ids, val_gt):

    candidates_by_s1 = (
        candidates_df.groupby("s1_entity_id")["candidate_entity_id"]
        .apply(set)
        .to_dict()
    )

    per_entity_recall = []
    skipped = []

    for s1_id in val_ids:
        if s1_id not in val_gt:
            skipped.append(s1_id)
            continue

        true_matches = val_gt[s1_id]
        candidate_ids = candidates_by_s1.get(s1_id, set())

        if len(true_matches) == 0:
            per_entity_recall.append(1.0)
        else:
            found = len(true_matches & candidate_ids)
            per_entity_recall.append(found / len(true_matches))

    macro_recall = sum(per_entity_recall) / len(per_entity_recall) if per_entity_recall else 0.0

    return {
        "macro_candidate_recall": macro_recall,
        "n_evaluated": len(per_entity_recall),
        "n_skipped_no_ground_truth": len(skipped),
    }

print("compute_candidate_recall() is defined and ready for the full-scale run.")


In [ ]:
val_ids = val_s1_ids
val_gt = val_gt_dict

print("Validation S1 IDs:", len(val_ids))
print("Ground-truth entries:", len(val_gt))
print("Corrupted entries:", len(corrupted_entries))

In [ ]:
print("First 5 validation S1 IDs:")
print(val_ids[:5])

print("\nFirst 5 ground-truth entries:")
for s1_id in list(val_gt.keys())[:5]:
    print(s1_id, "→", val_gt[s1_id])

In [ ]:
VALIDATION_S1_IDS = set(val_ids)

s1_val = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s1_val = s1_val[
    s1_val["entity_id"].isin(VALIDATION_S1_IDS)
].copy()

print("Validation S1 rows loaded:", len(s1_val))

In [ ]:
s1_val["name_norm"] = (
    s1_val["business_name"]
    .map(normalize_text)
)

print(
    "Validation S1 records with non-empty names:",
    (s1_val["name_norm"] != "").sum()
)

In [ ]:
def build_full_name_index(path, chunk_size=200_000):

    index = defaultdict(set)

    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name"],
        chunksize=chunk_size
    ):

        total_rows += len(chunk)

        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_text)
        )

        chunk = chunk[chunk["name_norm"] != ""]

        for name, group in chunk.groupby("name_norm"):

            index[name].update(
                group["entity_id"].tolist()
            )

        print(
            f"Processed {total_rows:,} rows..."
        )

    return index

In [ ]:
start = perf_counter()

s2_full_name_index = build_full_name_index(
    S2_PATH,
    chunk_size=200_000
)

elapsed = perf_counter() - start

print("\nS2 index built.")
print("Unique normalized names:", len(s2_full_name_index))
print(f"Time: {elapsed:.2f} seconds")

In [ ]:
start = perf_counter()

s3_full_name_index = build_full_name_index(
    S3_PATH,
    chunk_size=200_000
)

elapsed = perf_counter() - start

print("\nS3 index built.")
print("Unique normalized names:", len(s3_full_name_index))
print(f"Time: {elapsed:.2f} seconds")

In [ ]:
validation_name_candidates = exact_name_candidates(
    s1_val,
    s2_full_name_index,
    s3_full_name_index
)

print(
    "Total exact-name candidate pairs:",
    len(validation_name_candidates)
)

In [ ]:
_ = candidate_stats(
    validation_name_candidates,
    label="Exact normalized-name block (validation)"
)

In [ ]:
exact_name_recall = compute_candidate_recall(
    validation_name_candidates,
    val_ids,
    val_gt
)

print(exact_name_recall)

Exact Normalized Name Blocking

Validation results:

- Candidate recall: 26.28%
- Total candidate pairs: 4,124,695
- Average candidates per S1: 13.97
- Median: 2
- P95: 78
- P99: 200
- Maximum: 459

In [ ]:
from collections import Counter

def token_frequency_full(path, chunk_size=200_000):
    frequency = Counter()
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["business_name"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        for name in chunk["business_name"]:
            normalized = normalize_text(name)

            if not normalized:
                continue

            tokens = set(normalized.split())
            frequency.update(tokens)

        print(f"Processed {total_rows:,} rows...")

    return frequency

In [ ]:
start = perf_counter()

s2_token_frequency = token_frequency_full(S2_PATH)

print("\nS2 token frequency built.")
print("Unique tokens:", len(s2_token_frequency))
print(f"Time: {perf_counter() - start:.2f} seconds")

In [ ]:
start = perf_counter()

s3_token_frequency = token_frequency_full(S3_PATH)

print("\nS3 token frequency built.")
print("Unique tokens:", len(s3_token_frequency))
print(f"Time: {perf_counter() - start:.2f} seconds")

In [ ]:
print("TOP S2 TOKENS")
for token, count in s2_token_frequency.most_common(30):
    print(repr(token), "→", count)

print("\nTOP S3 TOKENS")
for token, count in s3_token_frequency.most_common(30):
    print(repr(token), "→", count)

In [ ]:
s2_token_frequency
s3_token_frequency

In [ ]:
thresholds = [100, 500, 1000, 5000, 10000, 25000, 50000]

print("S2 tokens surviving each threshold")
print("-" * 50)

for threshold in thresholds:
    count = sum(
        freq <= threshold
        for freq in s2_token_frequency.values()
    )
    print(f"<= {threshold:>6}: {count:,} tokens")

print("\nS3 tokens surviving each threshold")
print("-" * 50)

for threshold in thresholds:
    count = sum(
        freq <= threshold
        for freq in s3_token_frequency.values()
    )
    print(f"<= {threshold:>6}: {count:,} tokens")

In [ ]:
def useful_tokens(name, frequency, threshold):
    normalized = normalize_text(name)

    if not normalized:
        return []

    return [
        token
        for token in set(normalized.split())
        if frequency.get(token, 0) <= threshold
    ]

In [ ]:
for threshold in thresholds:
    usable = 0
    total_tokens = 0

    for name in s1_val["business_name"]:
        tokens = useful_tokens(
            name,
            s2_token_frequency,
            threshold
        )

        if tokens:
            usable += 1
            total_tokens += len(tokens)

    print(
        f"Threshold {threshold:>6}: "
        f"{usable:,} / {len(s1_val):,} S1 records have "
        f"at least one usable token | "
        f"avg usable tokens = "
        f"{total_tokens / len(s1_val):.2f}"
    )

In [ ]:
TOKEN_FREQ_THRESHOLD = 25_000

def build_filtered_token_index(
    path,
    token_frequency,
    max_frequency=25_000,
    chunk_size=200_000
):
    index = defaultdict(set)

    total_rows = 0
    kept_tokens = 0

    allowed_tokens = {
        token
        for token, freq in token_frequency.items()
        if freq <= max_frequency
    }

    print("Allowed tokens:", len(allowed_tokens))

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        for entity_id, name in zip(
            chunk["entity_id"],
            chunk["business_name"]
        ):
            normalized = normalize_text(name)

            if not normalized:
                continue

            tokens = set(normalized.split())

            for token in tokens:
                if token in allowed_tokens:
                    index[token].add(entity_id)

        print(f"Processed {total_rows:,} rows...")

    print(
        f"Finished. Index contains {len(index):,} tokens."
    )

    return index


In [ ]:
start = perf_counter()

s2_filtered_token_index = build_filtered_token_index(
    S2_PATH,
    s2_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS2 filtered token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

In [ ]:
start = perf_counter()

s3_filtered_token_index = build_filtered_token_index(
    S3_PATH,
    s3_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS3 filtered token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

In [ ]:
def evaluate_filtered_token_block(
    s1_df,
    s2_index,
    s3_index,
    val_gt,
    max_s1=None
):

    total_candidates = 0
    s1_with_candidates = 0

    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if max_s1 is not None and i >= max_s1:
            break

        s1_id = row["entity_id"]

        normalized = normalize_text(row["business_name"])

        if not normalized:
            candidate_ids = set()
        else:
            tokens = set(normalized.split())

            candidate_ids = set()

            for token in tokens:
                candidate_ids.update(
                    s2_index.get(token, set())
                )
                candidate_ids.update(
                    s3_index.get(token, set())
                )

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates

        if n_candidates > 0:
            s1_with_candidates += 1

        per_s1_counts.append(n_candidates)

        # Recall
        if s1_id not in val_gt:
            skipped += 1
        else:
            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:
                recall_sum += 1.0
            else:
                found = len(
                    true_matches & candidate_ids
                )

                recall_sum += (
                    found / len(true_matches)
                )

            evaluated += 1

        if (i + 1) % 10_000 == 0:
            print(
                f"Processed {i + 1:,} S1 records | "
                f"candidates so far: {total_candidates:,}"
            )

    import numpy as np

    counts = np.array(per_s1_counts)

    result = {
        "total_candidate_pairs": total_candidates,
        "s1_records_with_candidates": s1_with_candidates,
        "average_candidates_per_s1": counts.mean(),
        "median_candidates_per_s1": np.median(counts),
        "p95_candidates_per_s1": np.quantile(counts, 0.95),
        "p99_candidates_per_s1": np.quantile(counts, 0.99),
        "max_candidates_per_s1": counts.max(),
        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated > 0 else 0.0
        ),
        "n_evaluated": evaluated,
        "n_skipped_no_ground_truth": skipped,
    }

    return result

In [ ]:
print(evaluate_filtered_token_block)

In [ ]:
start = perf_counter()

token_test_10k = evaluate_filtered_token_block(
    s1_df=s1_val,
    s2_index=s2_filtered_token_index,
    s3_index=s3_filtered_token_index,
    val_gt=val_gt,
    max_s1=10_000
)

print("\n10K TOKEN BLOCK TEST")
print("-" * 50)

for key, value in token_test_10k.items():
    print(f"{key}: {value}")

print(f"\nTime: {perf_counter() - start:.2f} seconds")

In [ ]:
del s2_filtered_token_index
del s3_filtered_token_index

import gc
gc.collect()

print("25k token indexes removed from memory.")

In [ ]:
TOKEN_FREQ_THRESHOLD = 5_000

print("Testing token frequency threshold:", TOKEN_FREQ_THRESHOLD)

In [ ]:
start = perf_counter()

s2_filtered_token_index = build_filtered_token_index(
    S2_PATH,
    s2_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS2 5k token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

In [ ]:
start = perf_counter()

s3_filtered_token_index = build_filtered_token_index(
    S3_PATH,
    s3_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS3 5k token index built in "
    f"{perf_counter() - start:.2f} seconds"
)